# 03 · Reproduce: saved results, a live rerun, noise and cost

**Goal:** check the shipped benchmark yourself.

What you need:
- `pip install .` (Python 3.12). Everything runs offline.
- About two minutes of compute for the live rerun (two workers).

Previous: [01 · Quickstart](01-quickstart.ipynb) · [02 · Geometry mode](02-geometry-mode.ipynb) · Saved results: [`benchmarks/`](../benchmarks/README.md)

## 1. Every dataset at a glance

The library ships one `BenchmarkResult` per dataset: ten seeds, every model, statistics, noise and cost.
Balanced accuracy (BA) is the headline metric; Δ is Qlassifier minus the other model, in percentage points, with the corrected 95% CI.

In [1]:
import pandas as pd

from qlassifier import BenchmarkResult, list_datasets

rows = []
for info in list_datasets():
    r = BenchmarkResult.cached(info.id)
    h = r.highlights
    q = r.model("qlassifier").metrics.balanced_accuracy
    d = h.vs_best_classical
    rows.append({
        "dataset": info.id,
        "Qlassifier BA": f"{q.mean:.3f} ± {q.std:.3f}",
        "Δ vs ZZ QSVC (pp)": f"{100 * h.vs_standard_qml.mean_diff:+.1f}",
        "best classical": r.model(h.best_classical).name,
        "Δ vs best classical (pp)": f"{100 * d.mean_diff:+.1f} [{100 * d.ci_low:+.1f}, {100 * d.ci_high:+.1f}]",
    })
pd.DataFrame(rows).set_index("dataset")

,Qlassifier BA,Δ vs ZZ QSVC (pp),best classical,Δ vs best classical (pp)
dataset,,,,
breast_cancer,0.948 ± 0.012,+37.6,MLP,"-1.3 [-3.1, +0.5]"
ionosphere,0.925 ± 0.018,+22.6,RBF-SVM,"-0.2 [-3.1, +2.7]"
sonar,0.824 ± 0.053,+27.9,MLP,"+0.5 [-8.5, +9.5]"
heart,0.816 ± 0.026,+28.3,Logistic regression,"-1.0 [-3.1, +1.2]"
protein_angles,0.898 ± 0.009,+35.5,Gradient boosting,"+1.0 [-0.2, +2.2]"


How to read it:
- Δ vs ZZ QSVC: large and positive everywhere.
- Δ vs best classical: an interval that contains zero is a statistical tie.

## 2. Rerun two seeds live

Every number depends only on the dataset, the seed, the model and the library versions, never on scheduling.
So a live rerun of seeds 0 and 1 must match the saved runs of those seeds.

In [2]:
from qlassifier import benchmark

live = benchmark("heart", seeds=2, n_jobs=2)
saved = BenchmarkResult.cached("heart")

ba_live = live.to_frame().set_index(["seed", "model"])["balanced_accuracy"]
ba_saved = saved.to_frame().set_index(["seed", "model"])["balanced_accuracy"].loc[ba_live.index]
print(f"{len(ba_live)} runs compared · largest |difference| = {(ba_live - ba_saved).abs().max():.1e}")

24 runs compared · largest |difference| = 0.0e+00


With the locked versions (`uv sync --locked`) the difference is zero.
Other patch versions of NumPy or scikit-learn can shift results in the last digits.

## 3. Noise

Part A runs Qlassifier on every dataset under shot noise and three IBM device noise models (1,024 shots, seeds 0–4).
The table shows accuracy lost against exact simulation, in percentage points. Negative means it gained.

In [3]:
records = []
for info in list_datasets():
    r = BenchmarkResult.cached(info.id)
    for s in r.noise.summary:
        if s.part == "A" and s.device != "exact":
            records.append({"dataset": info.id, "device": s.device, "accuracy lost (pp)": round(s.drop_mean_pp, 2)})
order = [info.id for info in list_datasets()]
devices = ["shots", "fake_torino", "fake_fez", "fake_brisbane"]
pd.DataFrame(records).pivot(index="dataset", columns="device", values="accuracy lost (pp)").reindex(index=order, columns=devices)

device,shots,fake_torino,fake_fez,fake_brisbane
dataset,,,,
breast_cancer,-0.12,0.12,0.12,0.12
ionosphere,-0.19,-0.38,-0.75,-0.38
sonar,-0.32,-0.32,-0.63,-0.63
heart,-1.56,-0.89,-1.33,-1.33
protein_angles,0.18,0.17,0.17,0.19


Part B compares quantum models on identical small Breast Cancer subsets, because fidelity kernels need one circuit per pair of rows:

In [4]:
bc = BenchmarkResult.cached("breast_cancer")
part_b = [
    {"model": s.model, "device": s.device, "accuracy lost (pp)": round(s.drop_mean_pp, 1)}
    for s in bc.noise.summary
    if s.part == "B" and s.device != "exact"
]
pd.DataFrame(part_b).pivot(index="model", columns="device", values="accuracy lost (pp)").reindex(columns=devices[1:])

device,fake_torino,fake_fez,fake_brisbane
model,,,
fqk_bandwidth_linear,22.5,20.0,17.5
pqk,0.0,1.7,0.0
qlassifier,0.8,0.8,0.8
separable,14.2,13.3,10.8


In [5]:
bc.plot("noise");

**Assumptions.** These are simulations with the noise models of IBM fake backends (Qiskit Aer).
They have no crosstalk, drift or leakage. Real-hardware runs are the next stage of the project.

## 4. Cost

Circuits to train and predict on Breast Cancer, qubits, and two-qubit gates per circuit after transpilation:

In [6]:
cost = pd.DataFrame([
    {
        "model": m.model,
        "qubits": m.qubits,
        "2Q gates (Torino)": m.two_qubit_gates["fake_torino"],
        "circuits: train + predict": m.circuits_fit_predict,
        "circuits: tuning on device": m.tuning_circuits_on_device,
    }
    for m in bc.cost.models
]).set_index("model")
cost

,qubits,2Q gates (Torino),circuits: train + predict,circuits: tuning on device
model,,,,
zz_qsvc,8,393,147061,0
fqk_bandwidth_full,8,393,147061,2276238
fqk_bandwidth_linear,8,54,147061,2276238
pqk,9,72,1707,0
separable,8,0,147061,2276238
qlassifier,4,0,1707,0


In [7]:
bc.plot("cost");

**Result.**
- Qlassifier needs 3 circuits per sample and none for tuning.
- Fidelity kernels need one circuit per pair of samples, and bandwidth tuning on a device multiplies that.

## 5. The full rerun

One command reproduces every number in the paper (it takes hours on a laptop):

```bash
qlassifier benchmark --noise
```

It writes `REPORT.md` (targets BT-1 to BT-10), `summary.md`, per-seed CSV files and plots.
The shipped copy lives in [`benchmarks/`](../benchmarks/README.md).